In [1]:
import pandas as pd

df = pd.read_csv("Crop_recommendation.csv")
df.head()

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [2]:
def soil_type(row):
    avg_npk = (row['N'] + row['P'] + row['K']) / 3
    
    if avg_npk < 40:
        return "Low Fertility"
    elif avg_npk < 80:
        return "Medium Fertility"
    else:
        return "High Fertility"

df['Soil_Type'] = df.apply(soil_type, axis=1)

In [3]:
def rainfall_category(rain):
    if rain < 100:
        return "Low"
    elif rain < 200:
        return "Medium"
    else:
        return "High"

df['Rainfall_Category'] = df['rainfall'].apply(rainfall_category)

In [4]:
df_final = df[['Soil_Type', 'Rainfall_Category', 'temperature', 'label']]

In [5]:
df_final['temperature'] = df['temperature'].round()

C:\Users\yadav\AppData\Local\Temp\ipykernel_1368\3796244264.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['temperature'] = df['temperature'].round()


In [6]:
X = df_final[['temperature', 'Soil_Type', 'Rainfall_Category']]
y = df_final['label']

X_encoded = pd.get_dummies(X, drop_first=True)

In [7]:
X_encoded.head()

,temperature,Soil_Type_Low Fertility,Soil_Type_Medium Fertility,Rainfall_Category_Low,Rainfall_Category_Medium
0,21.0,False,True,False,False
1,22.0,False,True,False,False
2,23.0,False,True,False,False
3,26.0,False,True,False,False
4,20.0,False,True,False,False


In [ ]:
# model_training.py
import pandas as pd
import numpy as np
import joblib
import os
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✅ All packages imported successfully!")

# ============================================================================
# DATA PREPARATION (EXACTLY AS YOU DID)
# ============================================================================

def prepare_crop_data():
    """
    Prepare the crop recommendation dataset exactly as you did
    """
    print("📊 Loading and preparing crop recommendation dataset...")
    
    # Load the dataset
    df = pd.read_csv("Crop_recommendation.csv")
    print(f"✅ Dataset loaded: {df.shape}")
    
    # Your exact soil_type function
    def soil_type(row):
        avg_npk = (row['N'] + row['P'] + row['K']) / 3
        if avg_npk < 40:
            return "Low Fertility"
        elif avg_npk < 80:
            return "Medium Fertility"
        else:
            return "High Fertility"
    
    df['Soil_Type'] = df.apply(soil_type, axis=1)
    
    # Your exact rainfall_category function
    def rainfall_category(rain):
        if rain < 100:
            return "Low"
        elif rain < 200:
            return "Medium"
        else:
            return "High"
    
    df['Rainfall_Category'] = df['rainfall'].apply(rainfall_category)
    
    # Select features as you did
    df_final = df[['Soil_Type', 'Rainfall_Category', 'temperature', 'label']]
    df_final['temperature'] = df['temperature'].round()
    
    # Features and target as you did
    X = df_final[['temperature', 'Soil_Type', 'Rainfall_Category']]
    y = df_final['label']
    
    # One-hot encoding with drop_first=True (exactly as you did)
    X_encoded = pd.get_dummies(X, drop_first=True)
    feature_columns = X_encoded.columns.tolist()
    
    print(f"\n📊 Data Overview:")
    print(f"   Total samples: {len(df_final)}")
    print(f"   Features after encoding: {X_encoded.shape[1]}")
    print(f"   Feature columns: {feature_columns}")
    print(f"   Target classes: {y.nunique()}")
    print(f"\n📊 Sample encoded data (as you got):")
    print(X_encoded.head())
    
    return X_encoded, y, feature_columns

# ============================================================================
# EVALUATION FUNCTION
# ============================================================================

def evaluate_model(true, predicted, model_name="", problem_type="multiclass"):
    """
    Evaluate model performance with multiple metrics
    """
    accuracy = accuracy_score(true, predicted)
    f1 = f1_score(true, predicted, average='weighted')
    
    # Confusion matrix
    cm = confusion_matrix(true, predicted)
    
    # Classification report
    report = classification_report(true, predicted, zero_division=0)
    report_dict = classification_report(true, predicted, output_dict=True, zero_division=0)
    
    return {
        'accuracy': accuracy,
        'f1_score': f1,
        'confusion_matrix': cm,
        'report': report,
        'report_dict': report_dict
    }

# ============================================================================
# MODEL TRAINING AND EVALUATION FUNCTION
# ============================================================================

def train_and_evaluate_models(X_train, X_test, y_train, y_test, problem_type="multiclass", 
                              include_models=None, verbose=True):
    """
    Train multiple models and evaluate their performance
    """
    
    # Define all available models (exactly as in your original code)
    all_models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "K-Neighbors": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "Gaussian NB": GaussianNB(),
        "SVM": SVC(kernel='rbf', probability=True, random_state=42),
        "XGBoost": XGBClassifier(eval_metric='mlogloss', random_state=42, n_jobs=-1),
        "CatBoost": CatBoostClassifier(verbose=False, random_state=42, thread_count=-1),
        "LightGBM": LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1),
        "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
    }
    
    # Filter models if specified
    if include_models:
        models = {name: all_models[name] for name in include_models if name in all_models}
    else:
        models = all_models
    
    results = {}
    trained_models = {}
    
    for name, model in models.items():
        if verbose:
            print(f"\n{'='*50}")
            print(f"🚀 Training {name}...")
            print(f"{'='*50}")
        
        try:
            # Fit model
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            
            # Evaluate
            metrics = evaluate_model(y_test, y_pred, name, problem_type)
            results[name] = metrics
            trained_models[name] = model
            
            if verbose:
                print(f"   ✅ Accuracy: {metrics['accuracy']:.4f}")
                print(f"   ✅ F1-Score: {metrics['f1_score']:.4f}")
            
        except Exception as e:
            if verbose:
                print(f"   ❌ Error with {name}: {str(e)}")
            results[name] = None
            trained_models[name] = None
    
    return results, trained_models

# ============================================================================
# RESULTS VISUALIZATION FUNCTIONS
# ============================================================================

def create_results_dataframe(results):
    """
    Create a pandas DataFrame from results for easy viewing
    """
    data = []
    for model_name, metrics in results.items():
        if metrics is not None:
            data.append({
                'Model': model_name,
                'Accuracy': metrics['accuracy'],
                'F1-Score': metrics['f1_score'],
            })
    
    df = pd.DataFrame(data)
    df = df.sort_values('Accuracy', ascending=False).reset_index(drop=True)
    return df

# ============================================================================
# MAIN TRAINING FUNCTION
# ============================================================================

def train_crop_model():
    """
    Main function to train and save the crop recommendation model
    """
    print("\n" + "="*60)
    print("CROP RECOMMENDATION MODEL TRAINING")
    print("="*60)
    
    # Prepare data using your exact method
    X_encoded, y, feature_columns = prepare_crop_data()
    
    # Encode target labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    
    print(f"\n🌾 Target classes: {label_encoder.classes_.tolist()}")
    print(f"   Number of classes: {len(label_encoder.classes_)}")
    
    # Split the data (using your train_test_split parameters)
    X_train, X_test, y_train, y_test = train_test_split(
        X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )
    
    print(f"\n📊 Train-Test Split:")
    print(f"   Train set: {X_train.shape}")
    print(f"   Test set: {X_test.shape}")
    
    # Train and evaluate models
    print("\n" + "="*60)
    print("STARTING MODEL TRAINING...")
    print("="*60)
    
    results, trained_models = train_and_evaluate_models(
        X_train, X_test, y_train, y_test, 
        problem_type="multiclass",
        verbose=True
    )
    
    # Display results
    print("\n" + "="*60)
    print("RESULTS SUMMARY")
    print("="*60)
    
    results_df = create_results_dataframe(results)
    print("\n📊 Model Performance Comparison:")
    print(results_df.to_string(index=False))
    
    # Find best model
    best_model_name = results_df.iloc[0]['Model']
    best_accuracy = results_df.iloc[0]['Accuracy']
    best_model = trained_models[best_model_name]
    
    print(f"\n🏆 Best Model: {best_model_name} with Accuracy: {best_accuracy:.4f}")
    
    # Create models directory
    os.makedirs('models', exist_ok=True)
    
    # Save with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Save the best model with all necessary components
    model_package = {
        'model': best_model,
        'model_name': best_model_name,
        'accuracy': best_accuracy,
        'label_encoder': label_encoder,
        'feature_columns': feature_columns,
        'crops': label_encoder.classes_.tolist(),
        'training_date': timestamp,
        'results_summary': results_df.to_dict('records')
    }
    
    # Save the model package
    model_filename = f"models/crop_model_{timestamp}.pkl"
    joblib.dump(model_package, model_filename)
    print(f"\n💾 Best model saved to: {model_filename}")
    
    # Save simplified version for app
    simple_model_filename = "models/crop_model.pkl"
    joblib.dump(model_package, simple_model_filename)
    print(f"💾 Model also saved to: {simple_model_filename} (for app use)")
    
    # Show sample input format based on your actual encoding
    print("\n📝 Expected input format for prediction (with drop_first=True):")
    sample_dict = {'temperature': 25}
    for col in feature_columns:
        if col != 'temperature':
            sample_dict[col] = False
    
    # Set example values
    if 'Soil_Type_Medium Fertility' in sample_dict:
        sample_dict['Soil_Type_Medium Fertility'] = True
    if 'Rainfall_Category_Medium' in sample_dict:
        sample_dict['Rainfall_Category_Medium'] = True
    
    sample_input = pd.DataFrame([sample_dict])
    print(sample_input.to_string(index=False))
    
    return model_package

# ============================================================================
# PREDICTION FUNCTION FOR TESTING (UPDATED FOR drop_first=True)
# ============================================================================

def test_prediction(model_package, temperature, soil_type, rainfall_category):
    """
    Test the model with sample inputs (updated for drop_first=True encoding)
    """
    if model_package is None:
        print("❌ No model loaded")
        return
    
    model = model_package['model']
    label_encoder = model_package['label_encoder']
    feature_columns = model_package['feature_columns']
    
    # Create input dictionary based on your encoding (with drop_first=True)
    input_dict = {'temperature': temperature}
    
    # For drop_first=True, we don't have all soil types and rainfall categories
    # We only have the ones that weren't dropped
    for col in feature_columns:
        if col != 'temperature':
            input_dict[col] = False
    
    # Set the correct values based on input
    # For soil type with drop_first=True, we need to know which columns exist
    soil_mapping = {
        'Low Fertility': 'Soil_Type_Low Fertility',
        'Medium Fertility': 'Soil_Type_Medium Fertility',
        'High Fertility': None  # This might be the dropped category
    }
    
    rainfall_mapping = {
        'Low': 'Rainfall_Category_Low',
        'Medium': 'Rainfall_Category_Medium',
        'High': None  # This might be the dropped category
    }
    
    # Set soil type
    soil_col = soil_mapping.get(soil_type)
    if soil_col and soil_col in input_dict:
        input_dict[soil_col] = True
    
    # Set rainfall category
    rainfall_col = rainfall_mapping.get(rainfall_category)
    if rainfall_col and rainfall_col in input_dict:
        input_dict[rainfall_col] = True
    
    # Convert to DataFrame
    input_df = pd.DataFrame([input_dict])
    
    # Ensure all columns are present
    input_df = input_df[feature_columns]
    
    print(f"\n🔮 Test Prediction Input:")
    print(input_df)
    
    # Make prediction
    prediction = model.predict(input_df)[0]
    probabilities = model.predict_proba(input_df)[0]
    
    # Get crop name
    crop_name = label_encoder.inverse_transform([prediction])[0]
    confidence = probabilities[prediction]
    
    print(f"\n🔮 Test Prediction Result:")
    print(f"   Input: Temp={temperature}°C, Soil={soil_type}, Rainfall={rainfall_category}")
    print(f"   Predicted Crop: {crop_name}")
    print(f"   Confidence: {confidence:.2%}")
    
    # Show top 3 recommendations
    top_indices = np.argsort(probabilities)[-3:][::-1]
    print("\n   Top 3 Recommendations:")
    for idx in top_indices:
        crop = label_encoder.inverse_transform([idx])[0]
        conf = probabilities[idx]
        print(f"      - {crop}: {conf:.2%}")
    
    return crop_name, confidence

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    # Train the model
    model_package = train_crop_model()
    
    # Test the model with some examples
    if model_package:
        print("\n" + "="*60)
        print("TESTING PREDICTIONS")
        print("="*60)
        
        # Test cases
        test_prediction(model_package, 25, "Medium Fertility", "Medium")
        test_prediction(model_package, 30, "High Fertility", "Low")
        test_prediction(model_package, 20, "Low Fertility", "High")
    
    print("\n✅ Training pipeline completed successfully!")

✅ Starting improved model training...

IMPROVED CROP RECOMMENDATION MODEL TRAINING

📊 Loading crop recommendation dataset...
✅ Dataset loaded: (2200, 8)
📊 Columns: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']

📊 Using 7 numerical features
🌾 Target classes: 22 crops

📊 Train-Test Split:
   Train set: (1760, 7)
   Test set: (440, 7)

🚀 Training Random Forest...
   Cross-validation accuracy: 0.9943 (+/- 0.0108)
   Test accuracy: 0.9955

🚀 Training Gradient Boosting...
   Cross-validation accuracy: 0.9739 (+/- 0.0127)
   Test accuracy: 0.9886

🚀 Training XGBoost...
   Cross-validation accuracy: 0.9903 (+/- 0.0085)
   Test accuracy: 0.9886

🚀 Training CatBoost...
